# Mathematical Tools for Machine Learning

Lecture 1 fitted a model without naming the mathematics it used. This notebook
names it. Four objects appear in every method for the rest of the term, and each
one gets a section here:

| object | the question it answers | section |
|---|---|---|
| Vectors and the design matrix | How is a data set an object algebra can act on? | 1 |
| Orthogonal projection | What does least squares actually do, geometrically? | 2 |
| Conditional probability | What does "given that" mean as an operation on data? | 3 |
| The gradient | Which way is downhill on a loss surface? | 4 |

Each section writes the calculation out and then checks it against something
independent — a geometric identity, a probability law, or a library routine. The
point of the checking is the habit: a claim you can verify two ways is a claim
you can trust.

The data is ISLP `Default`: 10,000 credit-card holders, each with a balance, an
income, and whether they defaulted. It is a stand-in here, since this notebook
is about the tools rather than about credit.

---

## 1. The design matrix

`Default.csv` sits beside this notebook. The cell below loads it and pulls out
two arrays:

- `X`, with one row per person and one column per measurement. This is the
  **design matrix** — the standard way a data set is handed to an algorithm.
- `y`, holding 1 for the people who defaulted and 0 for everyone else. Comparing
  a text column against `'Yes'` gives booleans, and `.to_numpy(float)` turns
  those into the numbers arithmetic needs.

Watch the shapes that print. Almost every bug in this course announces itself as
a shape mismatch first.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (enables 3-D axes)

# The course palette, so the notebook figures match the slides.
BLUE, CLAY, SAGE, MUTED = '#00356B', '#7A3E47', '#5B7864', '#66727E'
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})

default = pd.read_csv('Default.csv')
X = default[['balance', 'income']].to_numpy(float)
y = (default['default'] == 'Yes').to_numpy(float)
print('X shape:', X.shape, 'y shape:', y.shape)
default.head()

### How long is a vector?

Before any geometry, one choice has to be made: what "length" means. Three
answers are used in this course, and they disagree.

$$\|v\|_1=\sum_j |v_j|, \qquad \|v\|_2=\sqrt{\sum_j v_j^2}, \qquad
  \|v\|_\infty=\max_j |v_j|.$$

The picture that separates them is the **unit ball** — every vector of length at
most one. The cell below draws all three for the plane, with one fixed vector
$v$ laid over each.

In [ ]:
v = np.array([0.8, 0.5])
t = np.linspace(0, 2 * np.pi, 400)
diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]])
square = np.array([[1, 1], [-1, 1], [-1, -1], [1, -1], [1, 1]])

panels = [(r'$\|v\|_1$', np.abs(v).sum(), diamond, None),
          (r'$\|v\|_2$', np.linalg.norm(v), None, (np.cos(t), np.sin(t))),
          (r'$\|v\|_\infty$', np.abs(v).max(), square, None)]

fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.6))
for ax, (label, value, poly, circle) in zip(axes, panels):
    shape = (poly[:, 0], poly[:, 1]) if poly is not None else circle
    ax.fill(*shape, color=BLUE, alpha=.12)
    ax.plot(*shape, color=BLUE, lw=1.6)
    ax.axhline(0, color=MUTED, lw=.6); ax.axvline(0, color=MUTED, lw=.6)
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=CLAY, lw=2))
    ax.text(v[0] + .05, v[1] + .05, '$v$', color=CLAY, fontsize=10)
    ax.set_title(f'{label} = {value:.2f}', fontsize=10)
    ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45); ax.set_aspect('equal')
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])
plt.show()

Same vector, three different numbers. The $\ell_1$ ball has corners on the
axes, the $\ell_2$ ball is round, and the $\ell_\infty$ ball is a square.

Those corners are not decoration. In Lecture 4 the lasso penalty is an
$\ell_1$-ball constraint, and the fact that its corners sit **on the axes** is
exactly why the lasso sets coefficients to zero while ridge, whose ball is
round, never quite does.

---

## 2. Projection and least squares

Least squares has a geometric description that explains its behaviour better
than the formula does.

The columns of $A$ span a plane inside $\mathbb{R}^{500}$ — every prediction the
model can possibly make lies in that plane. The response $y$ almost certainly
does not. Least squares picks the point of the plane closest to $y$, which is
the **orthogonal projection** of $y$ onto it. "Closest" and "orthogonal" are the
same condition, and the cell below checks it directly.

**Reading the code.** The first three lines cut down to 500 rows and standardize
them, exactly as in Lecture 1, then prepend a column of ones for the intercept.
`np.linalg.solve` solves the normal equations $A^\top A\beta = A^\top y$.

The interesting line is the assertion. If the fit really is a projection, the
**residual** $y - A\beta$ must be perpendicular to every column of $A$, so
$A^\top(y - A\beta)$ must be the zero vector. That is a fact about geometry, and
it holds no matter what the data is — which makes it a genuine check on the
arithmetic rather than a restatement of it. `atol=1e-10` allows for
floating-point rounding.

In [ ]:
# Use 500 rows so the geometry is easy to inspect.
X_small = X[:500]
y_small = y[:500]
mu, sd = X_small.mean(0), X_small.std(0)
A = np.column_stack([np.ones(len(X_small)), (X_small - mu) / sd])

beta = np.linalg.solve(A.T @ A, A.T @ y_small)
fitted = A @ beta
residual = y_small - fitted
print('beta:', beta)
print('A.T @ residual:', A.T @ residual)
assert np.allclose(A.T @ residual, 0, atol=1e-10)

The assertion passed, so the geometry holds. Here is what it looks like.

With one column the column space is a **line**, and the fit is the point of that
line nearest $y$. With two columns it is a **plane**, and the fit is the point of
the plane nearest $y$. In both cases the residual drops onto the space at a
right angle, which is the assertion above, drawn.

In [ ]:
fig = plt.figure(figsize=(7.6, 3.2))

# --- one column: projection onto a line ---
ax = fig.add_subplot(1, 2, 1)
a, w = np.array([3., 1.]), np.array([1.4, 2.8])
proj = (w @ a) / (a @ a) * a
resid = w - proj
ax.plot([-.4, 3.7], [-.4 / 3, 3.7 / 3], color=MUTED, lw=1, ls='--')
for tip, color, width in [(a, MUTED, 1.5), (w, BLUE, 2.2), (proj, SAGE, 2.2)]:
    ax.annotate('', xy=tip, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=width))
ax.annotate('', xy=w, xytext=proj, arrowprops=dict(arrowstyle='->', color=CLAY, lw=2.2))
u, e = resid / np.linalg.norm(resid) * .20, a / np.linalg.norm(a) * .20
ax.plot([proj[0] + u[0], proj[0] + u[0] + e[0], proj[0] + e[0]],
        [proj[1] + u[1], proj[1] + u[1] + e[1], proj[1] + e[1]], color=CLAY, lw=1.1)
ax.text(w[0] + .05, w[1] + .10, '$y$', color=BLUE, fontsize=11)
ax.text(proj[0] + .16, proj[1] - .30, r'$\widehat{y}$', color=SAGE, fontsize=11)
ax.text(2.55, .55, 'column space', color=MUTED, fontsize=8.5, rotation=18)
ax.set_xlim(-.4, 4.0); ax.set_ylim(-.5, 3.3); ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([]); ax.set_title('One column: a line', fontsize=10)

# --- two columns: projection onto a plane ---
ax = fig.add_subplot(1, 2, 2, projection='3d')
grid = np.linspace(-.15, 1.25, 2)
GX, GY = np.meshgrid(grid, grid)
ax.plot_surface(GX, GY, np.zeros_like(GX), alpha=.16, color=BLUE, edgecolor='none')
y3 = np.array([.7, .5, 1.1])
p3 = np.array([y3[0], y3[1], 0])
for column in (np.array([1., 0, 0]), np.array([0, 1., 0])):
    ax.quiver(0, 0, 0, *column, color=MUTED, lw=1.4, arrow_length_ratio=.12)
ax.quiver(0, 0, 0, *y3, color=BLUE, lw=2.2, arrow_length_ratio=.12)
ax.quiver(0, 0, 0, *p3, color=SAGE, lw=2.2, arrow_length_ratio=.14)
ax.plot(*zip(p3, y3), color=CLAY, lw=2.2)
ax.text(y3[0], y3[1], y3[2] + .12, '$y$', color=BLUE, fontsize=11)
ax.text(p3[0] + .05, p3[1], -.20, r'$\widehat{y}$', color=SAGE, fontsize=11)
ax.text(.78, .52, .55, 'residual', color=CLAY, fontsize=8.5)
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.set_title('Two columns: a plane', fontsize=10)
ax.view_init(elev=18, azim=-58); ax.set_box_aspect((1, 1, .85))
plt.show()

---

## 3. Eigenvalues, and what a matrix does to space

A matrix is a map. The cleanest way to see what a particular matrix *does* is to
feed it every unit vector at once — the unit circle — and look at the shape that
comes out. It is always an **ellipse**, and the two questions that matter are
how long its axes are and which way they point.

For a **symmetric** matrix $S$ the answer is the eigendecomposition: there are
directions $v$ that $S$ leaves alone except for stretching, $Sv=\lambda v$, and
those directions are the axes of the ellipse.

For a **general** matrix $M$ no direction need survive, and the singular value
decomposition answers the question anyway: it names an input direction $v_k$ and
an output direction $u_k$ with $Mv_k=\sigma_k u_k$.

In [ ]:
t = np.linspace(0, 2 * np.pi, 300)
circle = np.vstack([np.cos(t), np.sin(t)])
S = np.array([[2.0, 0.8], [0.8, 1.0]])      # symmetric
M = np.array([[1.6, 1.1], [-0.4, 0.9]])     # general

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.5))

# --- symmetric: eigenvectors keep their direction ---
w, V = np.linalg.eigh(S)
order = np.argsort(w)[::-1]
w, V = w[order], V[:, order]
ax = axes[0]
ax.plot(*circle, color=MUTED, lw=1, ls='--')
ax.plot(*(S @ circle), color=BLUE, lw=1.8)
for k in range(2):
    v = V[:, k]
    if v[0] < 0:
        v = -v
    ax.annotate('', xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.5))
    ax.annotate('', xy=w[k] * v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=CLAY, lw=2.3))
    perp = np.array([-v[1], v[0]]) * .30
    ax.text(*(w[k] * v * .62 + perp), f'$\\lambda_{k+1}={w[k]:.2f}$',
            color=CLAY, fontsize=9.5, ha='center')
ax.set_title('Symmetric $S$: eigenvectors keep their direction', fontsize=9.5)

# --- general: the SVD names an input and an output direction ---
U, sv, Vt = np.linalg.svd(M)
ax = axes[1]
ax.plot(*circle, color=MUTED, lw=1, ls='--')
ax.plot(*(M @ circle), color=BLUE, lw=1.8)
for k in range(2):
    v, u = Vt[k], U[:, k]
    if v[0] < 0:
        v, u = -v, -u
    ax.annotate('', xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.5))
    ax.text(*(v * 1.26), f'$v_{k+1}$', color=MUTED, fontsize=9.5, ha='center', va='center')
    ax.annotate('', xy=sv[k] * u, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=SAGE, lw=2.3))
    ax.text(*(sv[k] * u * 1.20), f'$\\sigma_{k+1}u_{k+1}$',
            color=SAGE, fontsize=9.5, ha='center', va='center')
ax.set_title('General $M$: the circle becomes an ellipse', fontsize=9.5)

for ax in axes:
    ax.set_aspect('equal'); ax.set_xlim(-3.1, 3.1); ax.set_ylim(-3.1, 3.1)
    ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
    ax.set_xticks([-2, 0, 2]); ax.set_yticks([-2, 0, 2])
plt.show()

print('eigenvalues of S: ', w.round(3))
print('singular values of M:', sv.round(3))

On the left, the clay arrows lie **along** the grey ones: $S$ stretched those two
directions by $2.44$ and $0.56$ and rotated neither. On the right no input
direction survives, so the SVD pairs each $v_k$ on the circle with a different
output direction $u_k$ on the ellipse.

The ratio of the axis lengths is the **condition number**. When it is large the
ellipse is a sliver, and every numerical problem in this course gets harder:
solving $A^\top A\beta=A^\top y$ becomes unstable (Lecture 4) and gradient descent
crawls (Lecture 5).

---

## 4. Gradients, curvature, and how to check a derivative

Optimization needs a direction to move in, and the gradient supplies it. Since
almost every gradient in this course is derived by hand, the useful skill is
being able to tell whether the one you derived is right.

**Reading the code.** `loss` evaluates
$\tfrac{1}{2n}\lVert A\beta - y\rVert^2$, and `gradient` returns the derivation
$\nabla = A^\top(A\beta - y)/n$, which is the formula Lecture 3 derives.

The loop is the check. The derivative of a function in the $j$-th coordinate is
the limit of

$$\frac{\text{loss}(\beta + \varepsilon e_j) - \text{loss}(\beta - \varepsilon e_j)}{2\varepsilon},$$

so evaluating that expression at a small $\varepsilon$ should reproduce the
$j$-th entry of the analytic gradient. `direction` is $\varepsilon e_j$: zeros
everywhere except a bump in coordinate $j$.

This is called a **finite-difference gradient check**, and it is the standard
way to catch a sign error or a missing factor in a hand-derived gradient. It is
far too slow to use for training — one loss evaluation per coordinate — which is
exactly why automatic differentiation exists, and that is the last section of
this lecture.

In [ ]:
def loss(beta):
    error = A @ beta - y_small
    return .5 * np.mean(error ** 2)

def gradient(beta):
    return A.T @ (A @ beta - y_small) / len(A)

candidate = np.array([.1, .2, -.1])
eps = 1e-6
finite_difference = np.empty_like(candidate)
for j in range(len(candidate)):
    direction = np.zeros_like(candidate); direction[j] = eps
    finite_difference[j] = (loss(candidate + direction) - loss(candidate - direction)) / (2 * eps)
print('analytic:', gradient(candidate))
print('finite difference:', finite_difference)
assert np.allclose(gradient(candidate), finite_difference, atol=1e-7)

In [ ]:
def loss_at(b1, b2, b0):
    error = A @ np.array([b0, b1, b2]) - y_small
    return .5 * np.mean(error ** 2)

grid1 = np.linspace(beta[1] - .20, beta[1] + .20, 140)
grid2 = np.linspace(beta[2] - .20, beta[2] + .20, 140)
G1, G2 = np.meshgrid(grid1, grid2)
Z = np.vectorize(lambda u, w: loss_at(u, w, beta[0]))(G1, G2)

start = np.array([beta[0], beta[1] - .14, beta[2] + .13])
g = gradient(start)
step = -g[1:] / np.linalg.norm(g[1:]) * .105

fig, ax = plt.subplots(figsize=(4.8, 3.5))
ax.contour(G1, G2, Z, levels=14, colors=BLUE, linewidths=.7, alpha=.5)
ax.plot(beta[1], beta[2], 'o', color=SAGE, ms=9)
ax.text(beta[1] + .012, beta[2] - .004, 'minimum', color=SAGE, fontsize=9)
ax.plot(start[1], start[2], 'o', color=CLAY, ms=7)
ax.annotate('', xy=(start[1] + step[0], start[2] + step[1]), xytext=(start[1], start[2]),
            arrowprops=dict(arrowstyle='-|>', color=CLAY, lw=2.4))
ax.text(start[1] - .055, start[2] + .016, r'$-\nabla L(\beta)$', color=CLAY, fontsize=10)
ax.set_xlabel(r'$\beta_{\rm balance}$'); ax.set_ylabel(r'$\beta_{\rm income}$')
ax.set_title('The negative gradient points downhill', fontsize=10)
plt.show()

The contours are the loss surface, sliced through the fitted intercept: each
ring is a set of coefficient pairs with the same loss. They are **ellipses
rather than circles**, because the two features are correlated, and that
elongation is what makes plain gradient descent slow — the steepest direction
points across the valley rather than along it. That single picture is most of
Lecture 5.

The arrow is $-\nabla L$ at a deliberately poor starting point. It points into
the rings, toward the minimum, which is what "downhill" means and what every
optimizer in this course is built on.

### Curvature: the second derivative decides the shape

The gradient says which way is downhill. The **Hessian**, the matrix of second
derivatives, says what the surface looks like once you get there — and by the
previous section, a symmetric matrix is fully described by its eigenvalues.

In [ ]:
grid = np.linspace(-2, 2, 200)
GX, GY = np.meshgrid(grid, grid)
cases = [('both $\\lambda>0$: a bowl', np.diag([3., 1.])),
         ('mixed signs: a saddle', np.diag([2., -2.])),
         ('very different sizes:\na long narrow valley', np.diag([12., .6]))]

fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.7))
for ax, (title, H) in zip(axes, cases):
    F = .5 * (H[0, 0] * GX ** 2 + 2 * H[0, 1] * GX * GY + H[1, 1] * GY ** 2)
    ax.contour(GX, GY, F, levels=np.linspace(F.min(), F.max(), 16),
               colors=BLUE, linewidths=.7, alpha=.65)
    ax.plot(0, 0, 'o', color=CLAY, ms=6)
    ax.set_title(title, fontsize=9); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

Three shapes, three sign patterns. A stationary point is a **minimum** only in
the first case; in the second it is a saddle, downhill in one direction and
uphill in another. The third is the one that matters most in practice: a genuine
minimum, but so elongated that the steepest direction points across the valley
rather than along it. That is the picture behind the ellipses in the previous
figure, and behind momentum in Lecture 5.

---

## 5. Probability: conditioning, and the Gaussian

Conditioning sounds abstract and is completely concrete: **keep the rows that
satisfy the condition, then recompute the proportion among those rows.** Every
conditional probability in this course is that operation.

Everything therefore depends on stating **what we condition on**. Here the
conditioning event is one specific, checkable statement about a customer:

$$A=\{\text{balance}\ \geq\ \text{the 90th percentile of balance}\},$$

which is the richest tenth of the balance distribution --- 1,000 of the 10,000
customers. The question is what defaulting looks like inside that group compared
with the population as a whole.

In [ ]:
balance = default['balance'].to_numpy(float)
cut = np.quantile(balance, .90)      # the 90th percentile of balance
event = balance >= cut               # the conditioning event A

p_marginal = y.mean()                # P(default)
p_conditional = y[event].mean()      # P(default | A) -- the same average, fewer rows

print(f'cut at the 90th percentile: balance = {cut:.0f}')
print(f'customers in the event:     {event.sum()} of {len(balance)}')
print(f'P(default)            = {p_marginal:.4f}')
print(f'P(default | balance>={cut:.0f}) = {p_conditional:.4f}')

Two lines of arithmetic, and the second is the first restricted to a subset of
the rows. The figure below is that sentence drawn: shade the event in the
histogram, then compare the outcome shares inside and outside it.

In [ ]:
n_all, n_event = len(balance), int(event.sum())

fig, axes = plt.subplots(1, 3, figsize=(9.9, 3.2))

# 1. the event we condition on, marked on the distribution of balance
ax = axes[0]
ax.hist(balance, bins=45, color=MUTED, alpha=.42)
ax.axvspan(cut, balance.max() * 1.02, color=BLUE, alpha=.16)
ax.axvline(cut, color=BLUE, lw=1.6)
ax.annotate('we condition on this:\n'
            f'balance ≥ {cut:.0f}\n'
            f'({n_event:,} customers)',
            xy=(cut, ax.get_ylim()[1] * .62),
            xytext=(cut - 1150, ax.get_ylim()[1] * .78), color=BLUE, fontsize=8.6,
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.3))
ax.set_xlabel('balance'); ax.set_ylabel('customers')
ax.set_title('1. Name the event', fontsize=10, color=BLUE)

# 2 and 3. the same two outcomes, counted over different sets of rows
for ax, share, n, head, sub in [
        (axes[1], p_marginal, n_all, '2. All customers',
         f'all {n_all:,} rows'),
        (axes[2], p_conditional, n_event, f'3. Only balance ≥ {cut:.0f}',
         f'{n_event:,} rows kept')]:
    bars = ax.bar([0, 1], [1 - share, share], color=[MUTED, CLAY], width=.6)
    for bar, value in zip(bars, [1 - share, share]):
        ax.text(bar.get_x() + bar.get_width() / 2, value + .025,
                f'{value:.3f}\n({round(value * n):,})', ha='center', fontsize=8.4)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['no default', 'default'])
    ax.set_ylim(0, 1.2); ax.set_title(head, fontsize=10, color=BLUE)
    ax.text(.5, -.30, sub, transform=ax.transAxes, ha='center',
            color=MUTED, fontsize=8.6)
axes[1].set_ylabel('share of the group')
plt.show()

Panels 2 and 3 have the same two categories and the same total height. The only
difference is which customers were counted: panel 2 uses all 10,000 rows, and
panel 3 uses the 1,000 rows that satisfy the event. The counts under each bar
show the renormalization directly --- 333 defaulters out of 10,000 becomes 269
out of 1,000. Restricting to the top
decile of balance takes the default rate from **0.033 to 0.269**, roughly an
eightfold increase, and that is the entire content of the word *given*.

Nothing stops us from doing this for every slice at once.

In [ ]:
edges = np.linspace(balance.min(), balance.max(), 26)
middles = (edges[:-1] + edges[1:]) / 2
counts, rates = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    inside = (balance >= lo) & (balance < hi)
    counts.append(inside.sum())
    rates.append(y[inside].mean() if inside.sum() >= 25 else np.nan)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.4, 4.2), sharex=True,
                               gridspec_kw={'height_ratios': [1, 1.25]})
ax1.bar(middles, counts, width=(edges[1] - edges[0]) * .9, color=MUTED, alpha=.45)
ax1.set_ylabel('customers')
ax1.set_title('Every bin is a conditioning event', fontsize=9.5)
ax2.plot(middles, rates, 'o-', color=CLAY, lw=1.8, ms=4)
ax2.axhline(p_marginal, color=BLUE, lw=1.4, ls='--')
ax2.text(edges[1], p_marginal + .03, f'$P(\\rm default)={p_marginal:.3f}$',
         color=BLUE, fontsize=8.5)
ax2.set_xlabel('balance'); ax2.set_ylabel(r'$P(\rm default\mid bin)$')
plt.show()

The top panel is the marginal distribution of balance, cut into bins. The bottom
panel gives the default rate inside each bin, with the overall rate as a dashed
line. Reading a single point off that curve is exactly the three-panel figure
above, done for one narrow slice.

The shape matters. The rate is flat and near zero up to a balance of about
1,300, then climbs steeply and passes 0.8. A curve that starts flat, rises
sharply, and flattens again is what logistic regression is built to fit, and it
arrives in Lecture 8.

### The Gaussian, and why its contours are ellipses

The multivariate Gaussian is the one distribution this course keeps returning
to, and it is built from exactly the objects above. Its density depends on the
data only through

$$(z-\mu)^\top \Sigma^{-1} (z-\mu),$$

a quadratic form in the symmetric matrix $\Sigma^{-1}$. Level sets of a quadratic
form are ellipses, and by the previous section their axes are the eigenvectors
of $\Sigma$, with lengths $\sqrt{\lambda_k}$.

In [ ]:
Z = (X - X.mean(0)) / X.std(0)
mu, C = Z.mean(0), np.cov(Z.T)
lam, Vc = np.linalg.eigh(C)

gx, gy = np.linspace(-3, 4, 220), np.linspace(-3.2, 3.2, 220)
GX, GY = np.meshgrid(gx, gy)
offset = np.dstack([GX, GY]) - mu
quad = np.einsum('...i,ij,...j->...', offset, np.linalg.inv(C), offset)
density = np.exp(-.5 * quad) / (2 * np.pi * np.sqrt(np.linalg.det(C)))

fig, ax = plt.subplots(figsize=(4.9, 3.6))
ax.scatter(Z[:, 0], Z[:, 1], s=3, alpha=.10, color=MUTED)
ax.contour(GX, GY, density, levels=6, colors=BLUE, linewidths=1.0)
for k in range(2):
    step = Vc[:, k] * np.sqrt(lam[k]) * 2
    ax.annotate('', xy=mu + step, xytext=mu,
                arrowprops=dict(arrowstyle='->', color=CLAY, lw=2.2))
    ax.text(*(mu + step * 1.08), f'$\\sqrt{{\\lambda_{k+1}}}$', color=CLAY, fontsize=9)
ax.set_xlabel('balance (standardized)'); ax.set_ylabel('income (standardized)')
ax.set_title('Contours are ellipses; the axes are eigenvectors of $\\Sigma$',
             fontsize=9.5)
plt.show()

print('covariance:\n', C.round(3))
print('eigenvalues:', lam.round(3))

The contours are nearly circular here because balance and income are almost
uncorrelated once standardized, so the two eigenvalues are close. Feed in two
strongly correlated features and the ellipse becomes a sliver pointing along
their shared direction — which is exactly what principal component analysis
looks for in Lecture 14.

Notice also that the Gaussian is a poor description of this cloud: income is
visibly bounded below and the mass is lopsided. Fitting one anyway, and then
checking, is the honest version of the workflow from Lecture 1.

---

## 6. The same fit, from a library

`fit_intercept=False` is there because $A$ already carries its own column of
ones. Agreement to rounding error means the projection we computed by hand is
the one `scikit-learn` computes.

In [ ]:
from sklearn.linear_model import LinearRegression

library = LinearRegression(fit_intercept=False).fit(A, y_small)
print('maximum coefficient difference:', np.max(np.abs(library.coef_ - beta)))
assert np.allclose(library.coef_, beta)

---

## What to take away

- A data set becomes a **design matrix**: rows are cases, columns are
  measurements, and a column of ones carries the intercept.
- Least squares is an **orthogonal projection**, and the residual being
  perpendicular to every column is a check you can run on any fit.
- A matrix turns the unit circle into an **ellipse**. Eigenvalues give its axes
  when the matrix is symmetric, singular values when it is not, and the ratio of
  those axes is what makes a problem well or badly conditioned.
- The gradient points downhill and the **Hessian** says what the surface looks
  like: a bowl, a saddle, or a long narrow valley.
- A hand-derived gradient should always be checked against finite differences
  before it is trusted.
- Conditioning is **filtering and renormalizing**: name the event, keep the rows
  that satisfy it, and take the same average over what is left.
- The Gaussian's contours are ellipses whose axes are the eigenvectors of the
  covariance, which ties the probability back to the algebra.

**Next.** Lecture 3 puts all four to work on a single problem: linear regression
with a design matrix, a squared-error loss, and the normal equations solved in
closed form.